In [ ]:
DIM = 20
MAX_EVALS = 350
N_RUNS = 16
BATCH_SIZE = 1
NUM_RESTARTS = 10
RAW_SAMPLES = 512
N_CANDIDATES = min(5000, max(2000, 200 * DIM)) 

# Botorch version

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from dataclasses import dataclass
import math

import torch
import gpytorch
import botorch

from src import test_functions
import jax
import jax.numpy as jnp
from src import gp, kernels, acquisition

import warnings
jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore", category=botorch.exceptions.BadInitialCandidatesWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
fun = jax.jit(test_functions.Ackley())

def torch_compatible_fun(x: torch.Tensor) -> torch.Tensor:
    return -fun(jnp.array(x))

# jax

In [ ]:
def jax_run(seed: int, use_matern: bool):
    # set random seeds
    np.random.seed(seed)
    torch.manual_seed(seed)

    # setup acquisition strategy and model
    strategy = acquisition.BFGS(NUM_RESTARTS)
    model = gp.GaussianProcess(
        kernel_metric=kernels.Euclidean(),
        kernel_profile=(
            kernels.Matern(5 / 2)
            if use_matern
            else kernels.SquaredExponential()
        ),
    )

    # initial samples
    sampler = torch.quasirandom.SobolEngine(DIM, scramble=True, seed=seed)
    x = sampler.draw(n=2 * DIM).to(dtype=torch.double)
    x = jnp.array(x)
    y = fun(x)

    # acquisition loop
    fit_time = 0.0
    acquisition_time = 0.0
    for step in (pbar := tqdm(range((MAX_EVALS - len(y)) // BATCH_SIZE))):
        # fit model
        start_time = time.time()
        model = model.fit(x, y)
        fit_time += time.time() - start_time

        # acquisition step
        start_time = time.time()
        x_next = strategy(model=model)
        acquisition_time += time.time() - start_time

        # evaluate new points and update model
        y_next = fun(x_next)
        x = jnp.concat((x, x_next), axis=0)
        y = jnp.concat((y, y_next), axis=0)

        # write on progressbar
        pbar.set_postfix(n=len(y), best_value=y.min().item())
    return x, y, acquisition_time, fit_time

In [ ]:
jax_matern_results = [jax_run(seed, use_matern=True) for seed in range(N_RUNS)]
jax_matern_results = [np.array(el) for el in zip(*jax_matern_results)]

In [ ]:
jax_squared_exponential_results = [jax_run(seed, use_matern=False) for seed in range(N_RUNS)]
jax_squared_exponential_results = [np.array(el) for el in zip(*jax_squared_exponential_results)]

# botorch

In [ ]:
def torch_run(seed: int):
    # set random seeds
    np.random.seed(seed)
    torch.manual_seed(seed)

    # initial samples
    sampler = torch.quasirandom.SobolEngine(DIM, scramble=True, seed=seed)
    x = sampler.draw(n=2 * DIM).to(dtype=torch.double)
    y = torch.tensor(torch_compatible_fun(x), dtype=torch.double).unsqueeze(-1)

    # acquisition loop
    fit_time = 0.0
    acquisition_time = 0.0
    for step in (pbar := tqdm(range((MAX_EVALS - len(y)) // BATCH_SIZE))):
        start_time = time.time()
        model = botorch.models.SingleTaskGP(
            train_X=x,
            train_Y=(y - y.mean()) / y.std(),
            likelihood=gpytorch.likelihoods.GaussianLikelihood(
                noise_constraint=gpytorch.constraints.Interval(1e-8, 1e-3)
            ),
        )
        botorch.fit.fit_gpytorch_mll(
            gpytorch.mlls.ExactMarginalLogLikelihood(model.likelihood, model)
        )
        fit_time += time.time() - start_time

        # Create a batch
        start_time = time.time()
        log_ei = botorch.acquisition.qLogExpectedImprovement(model, y.max())
        x_next, acq_value = botorch.optim.optimize_acqf(
            acq_function=botorch.acquisition.qLogExpectedImprovement(model, y.max()),
            bounds=torch.stack([torch.zeros(DIM), torch.ones(DIM)]).to(
                dtype=torch.double
            ),
            q=BATCH_SIZE,
            num_restarts=NUM_RESTARTS,
            raw_samples=RAW_SAMPLES,
        )
        acquisition_time += time.time() - start_time

        # Evaluate new points and update model
        y_next = torch.tensor(
            torch_compatible_fun(x_next), dtype=torch.double
        ).unsqueeze(-1)
        x = torch.cat((x, x_next), axis=0)
        y = torch.cat((y, y_next), axis=0)

        # write on progressbar
        pbar.set_postfix(n=len(y), best_value=y.max().item())

    x, y = x.cpu().numpy(), y.cpu().numpy().squeeze(-1)
    return x, -y, acquisition_time, fit_time

In [ ]:
torch_results = [torch_run(seed) for seed in range(N_RUNS)]
torch_results = [np.array(el) for el in zip(*torch_results)]

# turbo

In [ ]:
@dataclass
class TurboState:
    length: float = 0.8
    length_min: float = 0.5**7
    length_max: float = 1.6
    failure_counter: int = 0
    success_counter: int = 0
    success_tolerance: int = 10  # Note: The original paper uses 3
    best_value: float = -float("inf")
    restart_triggered: bool = False

    @property
    def failure_tolerance(self):
        return math.ceil(max([4.0 / BATCH_SIZE, float(DIM) / BATCH_SIZE]))

    def update(self, y: torch.Tensor):
        if max(y) > self.best_value + 1e-3 * math.fabs(self.best_value):
            self.success_counter += 1
            self.failure_counter = 0
        else:
            self.success_counter = 0
            self.failure_counter += 1

        if self.success_counter == self.success_tolerance:  # Expand trust region
            self.length = min(2.0 * self.length, self.length_max)
            self.success_counter = 0
        elif self.failure_counter == self.failure_tolerance:  # Shrink trust region
            self.length /= 2.0
            self.failure_counter = 0

        self.best_value = max(self.best_value, max(y).item())
        if self.length < self.length_min:
            self.restart_triggered = True
        return self

def turbo_run(seed: int, use_matern: bool = True):
    # set random seeds
    np.random.seed(seed)
    torch.manual_seed(seed)

    # initial samples
    sampler = torch.quasirandom.SobolEngine(DIM, scramble=True, seed=seed)
    x = sampler.draw(n=2 * DIM).to(dtype=torch.double)
    y = torch.tensor(torch_compatible_fun(x), dtype=torch.double).unsqueeze(-1)

    # acquisition loop
    fit_time = 0.0
    acquisition_time = 0.0
    state = TurboState(best_value=max(y).item())
    for step in (pbar := tqdm(range((MAX_EVALS - len(y)) // BATCH_SIZE))):
        assert x.min() >= 0.0
        assert x.max() <= 1.0
        assert torch.all(torch.isfinite(y))

        # Do the fitting and acquisition function optimization inside the Cholesky context
        with gpytorch.settings.max_cholesky_size(float("inf")):
            start_time = time.time()
            model = botorch.models.SingleTaskGP(
                train_X=x,
                train_Y=(y - y.mean()) / y.std(),
                covar_module=gpytorch.kernels.ScaleKernel(
                    gpytorch.kernels.MaternKernel(
                        nu=2.5,
                        ard_num_dims=DIM,
                        lengthscale_constraint=gpytorch.constraints.Interval(0.005, 4.0),
                    )
                ) if use_matern else None,
                likelihood=gpytorch.likelihoods.GaussianLikelihood(
                    noise_constraint=gpytorch.constraints.Interval(1e-8, 1e-3)
                ),
            )
            botorch.fit.fit_gpytorch_mll(
                gpytorch.mlls.ExactMarginalLogLikelihood(model.likelihood, model)
            )
            fit_time += time.time() - start_time

            # Scale the TR to be proportional to the lengthscales
            start_time = time.time()
            x_center = x[y.argmax(), :].clone()
            weights = model.covar_module.base_kernel.lengthscale.squeeze().detach()
            weights = weights / weights.mean()
            weights = weights / torch.prod(weights.pow(1.0 / len(weights)))
            tr_lb = torch.clamp(x_center - weights * state.length / 2.0, 0.0, 1.0)
            tr_ub = torch.clamp(x_center + weights * state.length / 2.0, 0.0, 1.0)

            pert = sampler.draw(N_CANDIDATES).to(dtype=torch.double)
            pert = tr_lb + (tr_ub - tr_lb) * pert

            # Create a perturbation mask
            prob_perturb = min(20.0 / DIM, 1.0)
            mask = torch.rand(N_CANDIDATES, DIM, dtype=torch.double) <= prob_perturb
            ind = torch.where(mask.sum(dim=1) == 0)[0]
            mask[ind, torch.randint(0, DIM - 1, size=(len(ind),))] = 1

            # Create candidate points from the perturbations and the mask
            x_cand = x_center.expand(N_CANDIDATES, DIM).clone()
            x_cand[mask] = pert[mask]

            # Sample on the candidate points
            thompson_sampling = botorch.generation.MaxPosteriorSampling(
                model=model, replacement=False
            )
            with torch.no_grad():  # We don't need gradients when using TS
                x_next = thompson_sampling(x_cand, num_samples=BATCH_SIZE)
            acquisition_time += time.time() - start_time

            # Evaluate new points and update model
            y_next = torch.tensor(
                torch_compatible_fun(x_next), dtype=torch.double
            ).unsqueeze(-1)
            state.update(y_next)
            x = torch.cat((x, x_next), axis=0)
            y = torch.cat((y, y_next), axis=0)

            # write on progressbar
            pbar.set_postfix(n=len(y), best_value=y.max().item())

    x, y = x.cpu().numpy(), y.cpu().numpy().squeeze(-1)
    return x, -y, acquisition_time, fit_time

In [ ]:
turbo_results = [turbo_run(seed, use_matern=True) for seed in range(N_RUNS)]
turbo_results = [np.array(el) for el in zip(*turbo_results)]

# plot

In [ ]:
plt.figure(figsize=(18, 6))
for i, (name, results) in enumerate(
    [
        ("botorch", torch_results),
        ("TuRBO", turbo_results),
        ("jax (Matern)", jax_matern_results),
        ("jax (RBF)", jax_squared_exponential_results),
    ]
):
    x, y, acquisition_time, fit_time = results

    runs, acquisitions = y.shape
    x = np.arange(acquisitions)
    y = np.minimum.accumulate(y, axis=-1)

    # ci on median
    qu = np.clip(0.5 + 1.96 * np.sqrt(0.25 / runs), 0, 1)
    ql = np.clip(0.5 - 1.96 * np.sqrt(0.25 / runs), 0, 1)
    ul = np.quantile(y, qu, axis=0)
    ll = np.quantile(y, ql, axis=0)

    plt.subplot(1, 2, 1)
    plt.plot(x, np.median(y, axis=0), label=name)
    plt.fill_between(x, ll, ul, alpha=0.2)

    plt.subplot(1, 4, 3)
    plt.boxplot(acquisition_time, positions=[i], patch_artist=True)

    plt.subplot(1, 4, 4)
    plt.boxplot(fit_time, positions=[i], patch_artist=True)

plt.subplot(1, 2, 1)
plt.plot([0, MAX_EVALS], [0.0, 0.0], "k--", lw=3, label="Optimum")
plt.xlabel("Function value", fontsize=18)
plt.xlabel("Number of evaluations", fontsize=18)
plt.title("20D Ackley")
plt.grid(visible=True)
plt.legend()

plt.subplot(1, 4, 3)
plt.xticks(range(4), ["botorch",  "TuRBO", "jax (Matern)", "jax (RBF)"])
plt.title("Acquisition time")
plt.ylabel("Time (seconds)")
plt.ylim(0, None)
plt.grid(True)

plt.subplot(1, 4, 4)
plt.xticks(range(4), ["botorch",  "TuRBO", "jax (Matern)", "jax (RBF)"])
plt.title("Model fitting time")
plt.ylabel("Time (seconds)")
plt.ylim(0, None)
plt.grid(True)

plt.tight_layout()
plt.show()